## **Feature Engineering for Multi-Output Regression**
We are now in the feature engineering phase of our project. The goal is to transform our per-pixel EBSD data into a single, aggregated set of microstructural features that can be used as the target variables ($y$) for our multi-output regression model.

Our process will involve:
1. **Data Filtering:** Cleaning the raw data by removing low-quality, un-indexed points.
2. **Feature Extraction:** Aggregating the per-pixel data to calculate key microstructural metrics like phase fraction, grain size, and average strain.
3. **Dataset Creation:** Combining these features to create a structured dataset for the neural network.


### **Setup and Data Loading**
We'll begin by importing our libraries and loading the EBSD data using the `load_ebsd_data` function we created in the EDA phase.


In [4]:
import pandas as pd
import numpy as np
import io
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.transform import Rotation as R
from skimage.measure import label, regionprops_table
import cv2

In [5]:
def load_ebsd_data(file_path):
    try:
        with open(file_path, 'r') as file:
            lines = file.readlines()

        header_info = {line.split(":")[0].strip().replace('# ', ''): line.split(":")[1].strip() for line in lines if line.startswith("# ") and ":" in line and not line.startswith('# COLUMN')}
        
        NCOLS_ODD = int(header_info.get('NCOLS_ODD', 0))
        NCOLS_EVEN = int(header_info.get('NCOLS_EVEN', 0))
        NROWS = int(header_info.get('NROWS', 0))

        header_end_index = None
        column_headers_line = None
        for i, line in enumerate(lines):
            if line.strip() == "# HEADER: End":
                header_end_index = i
            if line.startswith("# COLUMN_HEADERS:"):
                column_headers_line = line
        
        if header_end_index is None or column_headers_line is None:
            raise ValueError("Could not find required header information.")

        column_headers = column_headers_line.split(":")[1].strip().split(",")
        column_headers = [header.strip() for header in column_headers]

        data_string = "".join(lines[header_end_index+1:])
        df = pd.read_csv(io.StringIO(data_string), sep='\s+', names=column_headers)
        
        return df, NCOLS_ODD, NCOLS_EVEN, NROWS

    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return None, None, None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None, None, None

file_path = "Hmg_NI-1000pts_txt.txt"
df, NCOLS_ODD, NCOLS_EVEN, NROWS = load_ebsd_data(file_path)

if df is not None:
    print("Data loaded successfully.")

Data loaded successfully.


### **Data Filtering and Cleaning**

As our EDA showed, there are data points with very low `CI` or high `Fit` values, which are likely artifacts or un-indexed pixels. We will filter out these low-quality points to ensure our feature extraction is based on reliable data. We'll use a threshold of `CI` > 0.1, as a `CI` of -1 is clearly bad data, and any non-zero `CI` indicates a low-confidence but still present signal.


In [6]:
if df is not None:
    print(f"Original number of data points: {len(df)}")
    df_filtered = df[df['CI'] > 0.1].copy()
    print(f"Number of data points after filtering low-CI pixels: {len(df_filtered)}")

Original number of data points: 84753
Number of data points after filtering low-CI pixels: 76165


### **Feature Extraction from EBSD Data**

We will now aggregate the filtered per-pixel EBSD data to create a single row of features for the entire sample. These will be our target variables ($y$) for the machine learning model.




#### **A. Calculate Phase Fraction**

We are calculating the percentage of the scanned area occupied by each phase. This is a direct measure of the phase distribution in the sample.

In [7]:
if df_filtered is not None:
    phase_counts = df_filtered['Phase index'].value_counts(normalize=True)
    phase_1_fraction = phase_counts.get(1, 0)
    phase_2_fraction = phase_counts.get(2, 0)
    print(f"\nPhase 1 (Iron Gamma) Fraction: {phase_1_fraction:.2%}")
    print(f"Phase 2 (Iron Epsilon) Fraction: {phase_2_fraction:.2%}")


Phase 1 (Iron Gamma) Fraction: 99.01%
Phase 2 (Iron Epsilon) Fraction: 0.99%


#### **B. Calculate Average Grain Size**

Calculating average grain size requires identifying individual grains from the data. A common method is to first create a binary image from a low-quality EBSD map (e.g., IQ map), where a low IQ value indicates a grain boundary. We'll use `scikit-image` to label and measure the properties of these grains.


In [8]:
def get_grain_size(df, NCOLS_ODD, NCOLS_EVEN, NROWS):
    # Recreate the 2D IQ map
    data_list = df['IQ'].tolist()
    data_array = np.zeros((NROWS, NCOLS_ODD), dtype=float)
    data_idx = 0
    for r in range(NROWS):
        num_cols = NCOLS_ODD if r % 2 == 0 else NCOLS_EVEN
        for c in range(num_cols):
            if c < NCOLS_ODD and data_idx < len(data_list):
                data_array[r, c] = data_list[data_idx]
            data_idx += 1
    
    # Binarize the IQ map to identify grain boundaries (low IQ)
    # We use a threshold to separate high IQ (grain interior) from low IQ (boundary)
    iq_threshold = np.percentile(data_array, 25) # Use the 25th percentile as a conservative threshold
    grain_boundaries = data_array < iq_threshold
    
    # Use labeling to find and measure grains
    labeled_grains = label(grain_boundaries == False)
    
    # Calculate grain areas and find the average
    regions = regionprops_table(labeled_grains, properties=['area'])
    grain_areas = pd.DataFrame(regions)
    average_grain_size = grain_areas['area'].mean()

    return average_grain_size

if df_filtered is not None:
    if NCOLS_ODD is not None and NROWS is not None:
        average_grain_size = get_grain_size(df_filtered, NCOLS_ODD, NCOLS_EVEN, NROWS)
        print(f"\nAverage Grain Size (in pixels): {average_grain_size:.2f}")


Average Grain Size (in pixels): 10606.67


#### **C. Calculate Overall Strain (KAM)**

The mean KAM value for the entire area is a robust metric for the overall plastic strain in the sample. We'll reuse our `calculate_kam` function to compute this value.


In [9]:
def euler_to_quaternion(phi1, PHI, phi2):
    rot = R.from_euler('ZXZ', [phi1, PHI, phi2], degrees=False)
    return rot.as_quat()

def quaternion_to_misorientation(q1, q2):
    w1, x1, y1, z1 = q1[3], q1[0], q1[1], q1[2]
    w2, x2, y2, z2 = q2[3], q2[0], q2[1], q2[2]

    mis_q_w = w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2
    mis_q_x = w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2
    mis_q_y = w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2
    mis_q_z = w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2
    
    mis_q = np.array([mis_q_x, mis_q_y, mis_q_z, mis_q_w])
    
    cos_theta_half = np.clip(np.abs(mis_q_w), 0, 1)
    angle = 2 * np.arccos(cos_theta_half) * 180 / np.pi
    return angle

def calculate_kam(df, NCOLS_ODD, NCOLS_EVEN, NROWS):
    quaternions = np.array([euler_to_quaternion(row['phi1'], row['PHI'], row['phi2']) for _, row in df.iterrows()])
    kam_values = np.zeros(len(df))

    data_idx_map = np.full((NROWS, NCOLS_ODD), -1, dtype=int)
    current_1d_idx = 0
    for r in range(NROWS):
        num_cols = NCOLS_ODD if r % 2 == 0 else NCOLS_EVEN
        for c in range(num_cols):
            if c < NCOLS_ODD:
                data_idx_map[r, c] = current_1d_idx
            current_1d_idx += 1
            if current_1d_idx >= len(quaternions):
                break
        if current_1d_idx >= len(quaternions):
            break

    for r in range(NROWS):
        num_cols = NCOLS_ODD if r % 2 == 0 else NCOLS_EVEN
        for c in range(num_cols):
            if c < NCOLS_ODD and data_idx_map[r, c] != -1:
                current_1d_idx = data_idx_map[r, c]
                current_quat = quaternions[current_1d_idx]
                neighbor_misorientations = []

                neighbor_offsets = [(-1, 0), (1, 0), (0, -1), (0, 1)]
                if r % 2 == 0:
                    neighbor_offsets.extend([(-1, -1), (1, -1)])
                else:
                    neighbor_offsets.extend([(-1, 1), (1, 1)])

                for dr, dc in neighbor_offsets:
                    neighbor_r, neighbor_c = r + dr, c + dc
                    
                    if 0 <= neighbor_r < NROWS and 0 <= neighbor_c < NCOLS_ODD:
                        neighbor_idx = data_idx_map[neighbor_r, neighbor_c]
                        if neighbor_idx != -1:
                            neighbor_quat = quaternions[neighbor_idx]
                            misorientation = quaternion_to_misorientation(current_quat, neighbor_quat)
                            neighbor_misorientations.append(misorientation)
                
                if neighbor_misorientations:
                    kam_values[current_1d_idx] = np.mean(neighbor_misorientations)
                else:
                    kam_values[current_1d_idx] = 0

    df['KAM'] = kam_values
    return df

if df_filtered is not None:
    df_with_kam = calculate_kam(df_filtered.copy(), NCOLS_ODD, NCOLS_EVEN, NROWS)
    average_kam = df_with_kam['KAM'].mean()
    print(f"Average Kernel Average Misorientation (KAM): {average_kam:.2f} degrees")

Average Kernel Average Misorientation (KAM): 101.48 degrees


### **Creating the Final Feature and Target Dataset**

We are now creating the final dataset, which would be used to train our multi-output regression model. For this project, we are assuming that nanoindentation P-h curve data exists for the same sample. We'll simulate this data for demonstration purposes, and then combine all the extracted features into a single DataFrame.

In a real-world scenario, you would replace the simulated data with actual nanoindentation metrics.


In [12]:
# Simulate nanoindentation data (our input features, X)
indentation_data = {
    'hardness': [4.5],  # Example: Hardness in GPa
    'elastic_modulus': [180.0], # Example: Elastic Modulus in GPa
    'p_max': [500], # Example: Maximum load in mN
    'h_max': [100]  # Example: Maximum depth in nm
}
X_simulated = pd.DataFrame(indentation_data)

# Aggregate EBSD features into a single row (our target variables, y)
y_targets = pd.DataFrame({
    'phase_1_fraction': [phase_1_fraction],
    'phase_2_fraction': [phase_2_fraction],
    'average_grain_size_pixels': [average_grain_size],
    'average_kam_degrees': [average_kam]
})

# Combine the simulated input features (X) and the extracted target variables (y)
final_dataset = pd.concat([X_simulated, y_targets], axis=1)

print("\nFinal Dataset for Multi-Output Regression:")
print(final_dataset)
print("We have successfully filtered the data and extracted key microstructural features, creating a single dataset ready for our neural network model. The next phase will be to design and train the model.")


Final Dataset for Multi-Output Regression:
   hardness  elastic_modulus  p_max  h_max  phase_1_fraction  \
0       4.5            180.0    500    100          0.990127   

   phase_2_fraction  average_grain_size_pixels  average_kam_degrees  
0          0.009873               10606.666667            101.48449  
We have successfully filtered the data and extracted key microstructural features, creating a single dataset ready for our neural network model. The next phase will be to design and train the model.
